# Group results

Loads every `_results.json` saved by the classifier pipeline and makes group-level plots.
Run this as subjects accumulate. Each new subject just needs notebook 01 run first.

In [ ]:
import sys
import os
from pathlib import Path

# Find repo root by walking up until we find config.yaml.
_here = Path('.').resolve()
repo_root = next(
    (p for p in [_here, _here.parent, _here.parent.parent]
     if (p / 'config.yaml').exists()),
    _here,
)
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
print(f'Working directory: {Path.cwd()}')

%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

from analysis.plots import load_all_results, CONDITION_COLORS

DERIVED_DIR = 'data/derived'
CLASSIFIER  = 'swlda'

plt.rcParams['figure.dpi'] = 120

In [ ]:
results = load_all_results(DERIVED_DIR)
print(f'Found {len(results)} processed subjects:')
for r in results:
    print(f'  sub-{r["subject_id"]}  ({r["classifier_type"]})')

if not results:
    raise RuntimeError('No results found. Run notebook 01 on at least one subject first.')

In [ ]:
# Build a tidy dataframe: one row per (subject, condition)
rows = []
for r in results:
    subj = r['subject_id']
    for cond, metrics in r['per_condition'].items():
        rows.append({
            'subject': subj,
            'condition': cond,
            'balanced_accuracy': metrics['balanced_accuracy'] * 100,
            'n_epochs': metrics['n_epochs'],
            'true_target_rate': metrics['true_target_rate'] * 100,
            'true_nontarget_rate': metrics['true_nontarget_rate'] * 100,
        })

df = pd.DataFrame(rows)
print(df.groupby('condition')['balanced_accuracy'].describe().round(1))

## Balanced accuracy by condition — all subjects

In [ ]:
conditions = df['condition'].unique()
subjects   = df['subject'].unique()
n_subj     = len(subjects)
n_cond     = len(conditions)

fig, ax = plt.subplots(figsize=(max(8, n_subj * 0.9), 5))

x = np.arange(n_subj)
bar_width = 0.8 / n_cond

for i, cond in enumerate(conditions):
    vals = [df[(df['subject'] == s) & (df['condition'] == cond)]['balanced_accuracy'].values
            for s in subjects]
    vals = [v[0] if len(v) > 0 else np.nan for v in vals]
    offset = (i - n_cond / 2 + 0.5) * bar_width
    ax.bar(x + offset, vals, width=bar_width,
           color=CONDITION_COLORS.get(cond, '#888888'),
           label=cond, edgecolor='white')

ax.axhline(50, color='black', linewidth=0.8, linestyle='--', label='Chance')
ax.set_xticks(x)
ax.set_xticklabels([f'sub-{s}' for s in subjects], rotation=45, ha='right')
ax.set_ylabel('Balanced accuracy (%)')
ax.set_ylim(0, 105)
ax.set_title(f'Balanced accuracy by condition — {n_subj} subjects  [{CLASSIFIER}]')
ax.legend(loc='lower right', fontsize=9)
fig.tight_layout()
plt.show()

## Group mean ± SD per condition

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

group = df.groupby('condition')['balanced_accuracy']
means = group.mean()
sds   = group.std()

cond_list = list(means.index)
colors    = [CONDITION_COLORS.get(c, '#888888') for c in cond_list]

ax.bar(cond_list, means.values, yerr=sds.values, capsize=5,
       color=colors, edgecolor='white')

# Overlay individual points
for i, cond in enumerate(cond_list):
    vals = df[df['condition'] == cond]['balanced_accuracy'].values
    ax.scatter([i] * len(vals), vals, color='black', s=20, zorder=5, alpha=0.6)

ax.axhline(50, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('Balanced accuracy (%)')
ax.set_ylim(0, 105)
ax.set_title(f'Group mean ± SD — n={n_subj} subjects')
fig.tight_layout()
plt.show()

## Subject × condition accuracy table

In [ ]:
table = df.pivot(index='subject', columns='condition', values='balanced_accuracy')
table = table.round(1)
table.loc['MEAN'] = table.mean().round(1)
table.loc['SD']   = table.iloc[:-1].std().round(1)
table

## True positive / true negative rates by condition

Useful for checking whether accuracy drops are driven by missed targets or false alarms.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, metric, label in zip(
    axes,
    ['true_target_rate', 'true_nontarget_rate'],
    ['True target rate (sensitivity)', 'True nontarget rate (specificity)'],
):
    group = df.groupby('condition')[metric]
    means = group.mean()
    sds   = group.std()
    cond_list = list(means.index)
    colors = [CONDITION_COLORS.get(c, '#888888') for c in cond_list]
    ax.bar(cond_list, means.values, yerr=sds.values, capsize=5,
           color=colors, edgecolor='white')
    for i, cond in enumerate(cond_list):
        vals = df[df['condition'] == cond][metric].values
        ax.scatter([i] * len(vals), vals, color='black', s=20, zorder=5, alpha=0.6)
    ax.axhline(50, color='black', linewidth=0.8, linestyle='--')
    ax.set_ylabel('%')
    ax.set_ylim(0, 105)
    ax.set_title(label)

fig.suptitle(f'Sensitivity / Specificity by condition — n={n_subj} subjects', fontsize=12)
fig.tight_layout()
plt.show()